# What this simulation is doing

This notebook runs a **conditioned 2D Navier-Stokes smoke** system (`ConditionedNavierStokes2D`).

- **State variables**: `smoke` (scalar), `u` (x-velocity), `v` (y-velocity).
- **Conditioning variable**: `buoyancy_y` controls vertical coupling from smoke to momentum.
- **Dynamics**: semi-Lagrangian advection + viscosity + buoyancy forcing + pressure projection.
- **Boundary conditions**: periodic in both spatial directions.

### How to read the three plotted channels
- **Smoke fades toward homogeneity** because there is no source term; diffusion (and some numerical diffusion from semi-Lagrangian advection) smooths gradients over long times.
- **Velocity starts at zero by design** (`u=v=0` initial condition), then buoyancy spins it up from smoke anomalies.
- **Velocity can stay active longer** while smoke still has residual structure; once smoke becomes nearly uniform, buoyancy weakens and viscosity damps flow.
- **Channel colors are independently normalized**, so apparent contrast in `smoke` vs `u`/`v` is not a direct magnitude comparison between channels.

This mirrors the PDEArena *conditioned* Navier-Stokes setup where trajectories are conditioned on buoyancy strength.

## Governing equations (conditioned smoke-flow)

We evolve an incompressible velocity field $\mathbf{u}=(u,v)$ and a smoke scalar $s$:
$$
\frac{\partial s}{\partial t} + \mathbf{u}\cdot\nabla s = \kappa \nabla^2 s,
$$
$$
\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u}\cdot\nabla)\mathbf{u}
= -\nabla p + \nu \nabla^2 \mathbf{u} + (0, \beta (s-\bar{s})),
\qquad \nabla\cdot\mathbf{u}=0.
$$

### Variable definitions

- $s(x,y,t)$: smoke/passive scalar field.
- $\mathbf{u}(x,y,t)=(u,v)$: velocity field.
- $p(x,y,t)$: pressure-like projection potential.
- $\nu$: kinematic viscosity.
- $\kappa$: smoke diffusivity.
- $\beta$: buoyancy coefficient (this is the conditioning parameter `buoyancy_y`).

In PDEArena's conditioned dataset generation, different trajectories use different fixed values of buoyancy strength.

In [ ]:
from autosim.experimental.simulations import ConditionedNavierStokes2D as Sim

sim = Sim(
    return_timeseries=True,
    log_level="warning",
    n=64,
    T=320.0,
    dt=0.1,
    nu=0.03,
    L=32.0,
    # L=1.0,
    cfl=0.35,
    snapshot_dt=1.0
    # parameters_range={
    #     "buoyancy_y": (0.2, 0.5),
    # },
)

data = sim.forward_samples_spatiotemporal(n=2, random_seed=42)

In [ ]:
# (batch, time, x, y, channels)
data["data"].shape

In [ ]:
from autosim.utils import plot_spatiotemporal_video

anim = plot_spatiotemporal_video(
    data["data"],
    batch_idx=1,
    channel_names=["smoke", "u", "v"],
)

In [ ]:
from IPython.display import HTML

HTML(anim.to_jshtml())

In [ ]:
# Sampled conditioning values used for this batch
data["constant_scalars"]